In [ ]:
!pip install plotly pandas reportlab

In [ ]:
import pandas as pd

# Using the direct link to the raw CSV file
df = pd.read_csv('https://gist.githubusercontent.com/SharmaAshwini/8bd642f6c46792a9c40c8ccad60391e9/raw/Sample_Superstore.csv', encoding='latin1')

df.head()

In [ ]:
# Convert date column
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Create Year-Month for trend
df['Year-Month'] = df['Order Date'].dt.to_period('M').astype(str)

df.info()

In [ ]:
total_revenue = df['Sales'].sum()
top_product = df.groupby('Product Name')['Sales'].sum().idxmax()
top_region = df.groupby('Region')['Sales'].sum().idxmax()

print("Total Revenue:", total_revenue)
print("Top Product:", top_product)
print("Top Region:", top_region)

In [ ]:
import plotly.express as px

# Revenue by Region
fig_region = px.bar(df.groupby('Region')['Sales'].sum().reset_index(),
                    x='Region', y='Sales',
                    title='Sales by Region')

fig_region.show()
top_products = df.groupby('Product Name')['Sales'].sum().nlargest(10).reset_index()

fig_products = px.bar(top_products,
                      x='Sales', y='Product Name',
                      orientation='h',
                      title='Top 10 Products')

fig_products.show()
# Sales Trend
trend = df.groupby('Year-Month')['Sales'].sum().reset_index()

fig_trend = px.line(trend,
                    x='Year-Month', y='Sales',
                    title='Sales Trend Over Time')

fig_trend.show()

In [ ]:
import ipywidgets as widgets#Used to create dropdown filters
from IPython.display import display

# Dropdowns
region_filter = widgets.Dropdown(options=df['Region'].unique(), description='Region:')
category_filter = widgets.Dropdown(options=df['Category'].unique(), description='Category:')

def update_dashboard(region, category):
    filtered = df[(df['Region'] == region) & (df['Category'] == category)]

    fig = px.bar(filtered.groupby('Sub-Category')['Sales'].sum().reset_index(),
                 x='Sub-Category', y='Sales',
                 title=f'Sales in {region} - {category}')

    fig.show()

widgets.interact(update_dashboard,
                 region=region_filter,
                 category=category_filter)

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph

doc = SimpleDocTemplate("/content/Sales_Report.pdf") # Changed path to local /content/

content = []

content.append(Paragraph(f"Total Revenue: {total_revenue}", None))
content.append(Paragraph(f"Top Product: {top_product}", None))
content.append(Paragraph(f"Top Region: {top_region}", None))

doc.build(content)

print("PDF Generated and saved to local Colab environment: Sales_Report.pdf")